# Chapter 15 — Threads and Locks

Python toolkit: `threading.Thread`, `Lock`/`RLock`, `Condition`, `Semaphore`, `Event`, `Barrier`, plus `concurrent.futures` and `asyncio`.

**GIL note:** in standard CPython only one thread runs Python bytecode at a time, so threads help with I/O, not CPU work. (Python 3.13+ ships an optional free-threaded build.) Race conditions still happen. The GIL can switch threads between bytecodes, so `x += 1` is **not** atomic.

A deadlock needs all four **Coffman conditions**: mutual exclusion, hold and wait, no preemption, and circular wait.

In [1]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

counter = 0


def unsafe_increment(n: int) -> None:
    global counter
    for _ in range(n):
        tmp = counter
        time.sleep(0)  # yield: makes the race visible
        counter = tmp + 1


threads = [threading.Thread(target=unsafe_increment, args=(200,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"expected 800, got {counter} (lost updates from the race)")

counter = 0
lock = threading.Lock()


def safe_increment(n: int) -> None:
    global counter
    for _ in range(n):
        with lock:
            tmp = counter
            time.sleep(0)
            counter = tmp + 1


threads = [threading.Thread(target=safe_increment, args=(200,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
assert counter == 800
print("with lock:", counter)

expected 800, got 227 (lost updates from the race)
with lock: 800


## 15.1 Thread vs. Process
A **process** is a running program with its own memory space, and processes communicate through IPC (pipes, sockets, shared memory). **Threads** live inside a process and share its heap, while each has its own stack and registers. Threads are cheaper to create and to switch between, but a shared heap needs synchronization. In Python, use `multiprocessing` / `ProcessPoolExecutor` for CPU-bound parallelism and threads for I/O-bound concurrency.

In [2]:
import os
from multiprocessing import get_context

shared = []


def append_pid() -> None:
    shared.append(os.getpid())


t = threading.Thread(target=append_pid); t.start(); t.join()
print("thread wrote into our list:", shared)  # same memory

ctx = get_context("fork") if hasattr(os, "fork") else get_context()
p = ctx.Process(target=append_pid); p.start(); p.join()
print("process wrote into ITS OWN copy; ours is unchanged:", shared)
assert len(shared) == 1

thread wrote into our list: [42052]


process wrote into ITS OWN copy; ours is unchanged: [42052]


## 15.2 Context Switch
How would you measure the time of a context switch? Ping-pong a token between two threads that block on each other. Each round trip forces two switches, so the per-switch cost is roughly `elapsed / (2 · rounds)`. That is an upper bound, since it includes the synchronization overhead.

In [3]:
def measure_context_switch(rounds: int = 5_000) -> float:
    a, b = threading.Semaphore(0), threading.Semaphore(0)

    def pong() -> None:
        for _ in range(rounds):
            a.acquire()
            b.release()

    t = threading.Thread(target=pong)
    t.start()
    start = time.perf_counter()
    for _ in range(rounds):
        a.release()
        b.acquire()
    elapsed = time.perf_counter() - start
    t.join()
    return elapsed / (2 * rounds)


print(f"~{measure_context_switch() * 1e6:.1f} µs per thread switch (upper bound)")

~3.3 µs per thread switch (upper bound)


## 15.3 Dining Philosophers
Five philosophers share five chopsticks, and each needs both neighbouring ones to eat. If every philosopher grabs the left chopstick first, they can deadlock. **Fix:** impose a global order on the chopsticks and always pick up the lower-numbered one first. That breaks the circular wait.

In [4]:
class Chopstick:
    def __init__(self, idx: int) -> None:
        self.idx = idx
        self.lock = threading.Lock()


class Philosopher(threading.Thread):
    def __init__(self, idx: int, left: Chopstick, right: Chopstick, bites: int) -> None:
        super().__init__()
        self.idx, self.bites, self.eaten = idx, bites, 0
        self.first, self.second = sorted((left, right), key=lambda c: c.idx)  # global ordering

    def run(self) -> None:
        for _ in range(self.bites):
            with self.first.lock, self.second.lock:
                self.eaten += 1
                time.sleep(0.0005)


N = 5
sticks = [Chopstick(i) for i in range(N)]
phils = [Philosopher(i, sticks[i], sticks[(i + 1) % N], bites=20) for i in range(N)]
for p in phils: p.start()
for p in phils: p.join(timeout=10)
assert all(p.eaten == 20 and not p.is_alive() for p in phils)
print("15.3 ok - no deadlock")

15.3 ok - no deadlock


## 15.4 Deadlock-Free Class
Design a class that hands out locks only if doing so can't cause a deadlock. Each process declares the order in which it will need its locks. We add those orders as edges in a directed graph and **reject any declaration that would create a cycle**. At runtime, a process may only acquire locks in the order it declared.

In [5]:
from collections import defaultdict


class LockFactory:
    def __init__(self, n_locks: int) -> None:
        self.locks = [threading.Lock() for _ in range(n_locks)]
        self.edges: dict[int, set[int]] = defaultdict(set)
        self.declared: dict[str, list[int]] = {}
        self._meta = threading.Lock()

    def _has_cycle(self, edges: dict[int, set[int]]) -> bool:
        WHITE, GREY, BLACK = 0, 1, 2
        color: dict[int, int] = defaultdict(int)

        def dfs(u: int) -> bool:
            color[u] = GREY
            for v in edges[u]:
                if color[v] == GREY or (color[v] == WHITE and dfs(v)):
                    return True
            color[u] = BLACK
            return False

        return any(color[u] == WHITE and dfs(u) for u in list(edges))

    def declare(self, pid: str, order: list[int]) -> bool:
        with self._meta:
            trial = defaultdict(set, {k: set(v) for k, v in self.edges.items()})
            for a, b in zip(order, order[1:]):
                trial[a].add(b)
            if self._has_cycle(trial):
                return False
            self.edges, self.declared[pid] = trial, order
            return True

    def acquire(self, pid: str, lock_id: int, held: list[int]) -> None:
        order = self.declared[pid]
        if order[len(held)] != lock_id:
            raise RuntimeError(f"{pid} must acquire locks in declared order {order}")
        self.locks[lock_id].acquire()
        held.append(lock_id)


lf = LockFactory(5)
assert lf.declare("A", [1, 2, 3, 4])
assert lf.declare("B", [1, 3, 4])
assert not lf.declare("C", [4, 1])   # 1->...->4 exists, 4->1 would make a cycle
held: list[int] = []
lf.acquire("B", 1, held)
try:
    lf.acquire("B", 4, held)
except RuntimeError as e:
    print("rejected:", e)
print("15.4 ok")

rejected: B must acquire locks in declared order [1, 3, 4]
15.4 ok


## 15.5 Call In Order
One `Foo` instance is shared by three threads. Thread A calls `first()`, B calls `second()`, and C calls `third()`, and they may start in any order. Make sure the calls always run first → second → third. Use two `Event`s as gates. (In Java you'd use semaphores; a Lock won't work, because it must be released by the thread that acquired it.)

In [6]:
import random


class Foo:
    def __init__(self) -> None:
        self.log: list[str] = []
        self._first_done = threading.Event()
        self._second_done = threading.Event()

    def first(self) -> None:
        self.log.append("first")
        self._first_done.set()

    def second(self) -> None:
        self._first_done.wait()
        self.log.append("second")
        self._second_done.set()

    def third(self) -> None:
        self._second_done.wait()
        self.log.append("third")


for _ in range(50):
    foo = Foo()
    calls = [foo.first, foo.second, foo.third]
    random.shuffle(calls)
    ts = [threading.Thread(target=c) for c in calls]
    for t in ts: t.start()
    for t in ts: t.join()
    assert foo.log == ["first", "second", "third"]
print("15.5 ok")

15.5 ok


## 15.6 Synchronized Methods
A class has synchronized method A and ordinary method B. With two threads and one instance, can they run A at the same time? **No**, because synchronized methods lock the instance. Can one run A while the other runs B? **Yes**, since B doesn't take the lock. With two *different* instances, both threads can run A at once. Demonstrated with a per-instance lock:

In [7]:
def synchronized(method):
    def wrapper(self, *a, **kw):
        with self._monitor:
            return method(self, *a, **kw)
    return wrapper


class Account:
    def __init__(self) -> None:
        self._monitor = threading.RLock()
        self.active = 0
        self.max_active = 0
        self._stat = threading.Lock()

    def _enter(self) -> None:
        with self._stat:
            self.active += 1
            self.max_active = max(self.max_active, self.active)

    def _exit(self) -> None:
        with self._stat:
            self.active -= 1

    @synchronized
    def a(self) -> None:
        self._enter(); time.sleep(0.05); self._exit()

    def b(self) -> None:
        self._enter(); time.sleep(0.05); self._exit()


def overlap(*calls) -> int:
    obj = calls[0].__self__
    ts = [threading.Thread(target=c) for c in calls]
    for t in ts: t.start()
    for t in ts: t.join()
    return obj.max_active


acct = Account(); assert overlap(acct.a, acct.a) == 1  # same instance, both synchronized: serialized
acct = Account(); assert overlap(acct.a, acct.b) == 2  # sync + non-sync: can overlap
x, y = Account(), Account()
start = time.perf_counter()
ts = [threading.Thread(target=x.a), threading.Thread(target=y.a)]
for t in ts: t.start()
for t in ts: t.join()
assert time.perf_counter() - start < 0.09  # serial would take 0.1s; different instances ran concurrently
print("15.6 ok")

15.6 ok


## 15.7 FizzBuzz (multithreaded)
Four threads: one prints "Fizz" for multiples of 3 only, one "Buzz" for multiples of 5 only, one "FizzBuzz" for multiples of 15, and one prints the number otherwise. They coordinate over a shared counter with a `Condition`. Each thread has a predicate that says whether it's its turn.

In [8]:
def threaded_fizzbuzz(n: int) -> list[str]:
    out: list[str] = []
    cond = threading.Condition()
    current = 1

    def worker(predicate, render) -> None:
        nonlocal current
        while True:
            with cond:
                cond.wait_for(lambda: current > n or predicate(current))
                if current > n:
                    return
                out.append(render(current))
                current += 1
                cond.notify_all()

    specs = [
        (lambda i: i % 15 == 0, lambda i: "FizzBuzz"),
        (lambda i: i % 3 == 0 and i % 5, lambda i: "Fizz"),
        (lambda i: i % 5 == 0 and i % 3, lambda i: "Buzz"),
        (lambda i: i % 3 and i % 5, str),
    ]
    ts = [threading.Thread(target=worker, args=s) for s in specs]
    for t in ts: t.start()
    for t in ts: t.join()
    return out


expected = ["FizzBuzz" if i % 15 == 0 else "Fizz" if i % 3 == 0 else "Buzz" if i % 5 == 0 else str(i) for i in range(1, 101)]
assert threaded_fizzbuzz(100) == expected
print(" ".join(threaded_fizzbuzz(15)))

1 2 Fizz 4 Buzz Fizz 7 8 Fizz Buzz 11 Fizz 13 14 FizzBuzz
